In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

# load + same split as before
df = pd.read_parquet("../data/sample.parquet")
df["Label"] = df["Label"].str.replace("�", "-", regex=False)
X = df.drop(columns=["Label", "is_attack"]); y = df["Label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)
b_fit, b_val = train_test_split(X_train[y_train == "BENIGN"], test_size=0.2, random_state=42)
true_bin = (y_test != "BENIGN").astype(int).values

# log-transform, train on benign only
Xl_fit  = np.log1p(b_fit.clip(lower=0))
Xl_val  = np.log1p(b_val.clip(lower=0))
Xl_test = np.log1p(X_test.clip(lower=0))

iso2 = IsolationForest(n_estimators=200, max_samples=1024, n_jobs=-1, random_state=42)
iso2.fit(Xl_fit)

s_test = -iso2.score_samples(Xl_test)
thr2 = np.percentile(-iso2.score_samples(Xl_val), 99)
flag2 = (s_test > thr2).astype(int)

print("ROC-AUC:", round(roc_auc_score(true_bin, s_test), 4))
print("Attack recall:", round(flag2[true_bin==1].mean(), 4),
      "| False alarm rate:", round(flag2[true_bin==0].mean(), 4))
print(pd.DataFrame({"Label": y_test.values, "flag": flag2})
        .groupby("Label")["flag"].mean().round(3).sort_values())

ROC-AUC: 0.9162
Attack recall: 0.0255 | False alarm rate: 0.0089
Label
DoS GoldenEye                 0.000
FTP-Patator                   0.000
Web Attack - Brute Force      0.000
SSH-Patator                   0.000
Web Attack - Sql Injection    0.000
DDoS                          0.002
PortScan                      0.003
DoS Hulk                      0.004
BENIGN                        0.009
Web Attack - XSS              0.015
Bot                           0.017
DoS slowloris                 0.191
Infiltration                  0.545
DoS Slowhttptest              0.596
Heartbleed                    1.000
Name: flag, dtype: float64


In [3]:
from sklearn.metrics import roc_curve
fpr, tpr, thr = roc_curve(true_bin, s_test)
for target in [0.01, 0.05, 0.10, 0.20]:
    i = np.searchsorted(fpr, target)
    print(f"False alarm rate {target:.0%}: attack recall {tpr[i]:.1%}")

False alarm rate 1%: attack recall 2.7%
False alarm rate 5%: attack recall 39.9%
False alarm rate 10%: attack recall 71.5%
False alarm rate 20%: attack recall 98.1%


In [4]:
import joblib
val_s = -iso2.score_samples(Xl_val)
thresholds = {"low": float(np.percentile(val_s, 99)),      # ~1% false alarms
              "medium": float(np.percentile(val_s, 95)),   # ~5%
              "high": float(np.percentile(val_s, 90))}     # ~10%
joblib.dump({"model": iso2, "features": list(X.columns), "thresholds": thresholds},
            "../src/iso_forest.joblib")
print(thresholds)

{'low': 0.6088504837572377, 'medium': 0.5548162500422529, 'high': 0.520125923930044}
